# Data Pre-Processing

## Libraries

We start by importing the necessary libraries. Pandas and NumPy for data manipulation, and statsmodels for statistical tests.

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from scipy.interpolate import CubicSpline

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler

import sys
sys.path.append('../')

from src.modwt import *

import pywt
from pywt import wavedec
from pywt import Wavelet

## Data Import

First, let's assume you've fetched your historical BTCUSDT price data into a CSV file, with columns 'date', 'open', 'high', 'low', 'close'.

In [2]:
ref_data_range = "1D"
data_name = 'ETHUSDT_1d' # BTCUSDT_1d, ETHUSDT_1d, SOLUSDT_1d
from_date = '2018-12-31 00:00:00' # 2020-12-31 00:00:00
until_date = '2020-12-31 00:00:00' # 2022-12-31 00:00:00
until_date_2 = '2021-01-02 00:00:00' # 2023-01-02 00:00:00

In [3]:
# Importing data
data = pd.read_csv(f'../data/{data_name}_data_from_20180101_to_20230102.csv', parse_dates=['date'], index_col='date')

In [4]:
data

,open,high,low,close,volume
date,,,,,
2018-01-02,733.01,763.55,716.80,754.99,53909.25885
2018-01-03,754.99,899.50,749.06,855.28,113257.78355
2018-01-04,855.13,950.01,810.00,934.03,89041.45688
2018-01-05,934.03,1009.72,890.01,940.00,102894.47384
2018-01-06,940.00,1045.00,930.00,959.30,97374.01630
...,...,...,...,...,...
2022-12-30,1190.15,1206.57,1186.77,1200.49,249130.48690
2022-12-31,1200.48,1202.15,1181.08,1199.99,266014.21150
2023-01-01,1199.98,1208.46,1191.66,1196.13,139379.28110


In [5]:
# # Initialize a DataFrame to hold all the transformations
# unified_data = pd.DataFrame(index=data.index)
# unified_data['original_close'] = data['close']

In [6]:
# Initial Plot
fig = px.line(data, x=data.index, y="close", title='Original BTCUSDT Close Price')
fig.show()

In [7]:
# Get data from_date until until_date inclusive
data = data[(data.index >= from_date) & (data.index <= until_date_2)]

In [8]:
data['original_close'] = data['close']

/var/folders/s0/_j22qz4n5j72grfwwy9xgbxc0000gn/T/ipykernel_1515/3471206824.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [9]:
data

,open,high,low,close,volume,original_close
date,,,,,,
2018-12-31,133.00,139.00,127.03,137.77,7.042594e+05,137.77
2019-01-01,137.81,138.66,128.70,131.45,4.846048e+05,131.45
2019-01-02,131.45,140.67,130.00,139.10,4.017769e+05,139.10
2019-01-03,139.10,155.37,137.76,152.01,9.414863e+05,152.01
2019-01-04,152.01,153.64,143.11,146.30,7.080913e+05,146.30
...,...,...,...,...,...,...
2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41
2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00
2020-12-31,732.00,758.74,716.62,752.17,1.008574e+06,752.17


In [10]:
# Initial Plot
fig = px.line(data, x=data.index, y="original_close", title='Original BTCUSDT Close Price')
fig.show()

## Handling Missing Values

We start by identifying and handling any missing values.

References: 

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.interpolate.html

https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html#scipy.interpolate.CubicSpline

In [11]:
# Check that there is no missing data
# data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [12]:
# Get all the dates in the start_date and end_date range
df_full_dates = pd.DataFrame()
df_full_dates['date'] = my_range

In [13]:
# show data['date'] values that are not in df_full_dates['date']
data[~data.index.isin(df_full_dates['date'])]

,open,high,low,close,volume,original_close
date,,,,,,


In [14]:
# Remove the data['date'] values that are not in df_full_dates['date']
data = data[data.index.isin(df_full_dates['date'])]

In [15]:
data_new = data.join(df_full_dates.set_index('date'), on='date', how='outer')
data_new.sort_values(by=['date'], inplace=True)

# data_new['date'] = pd.to_datetime(data_new['date'])  # Convert 'date' column to datetime type if needed
data = data_new.copy()  # Make a copy of the dataframe
# data = data.set_index('date')  # Set 'date' as the index for convenient time-based operations
# data_filled = data_new.fillna(method='ffill')
# data_filled = data_new.reset_index()

In [16]:
my_range_new = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range_new[~my_range_new.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

### Add `log_return` column

In [17]:
# add log return
data['processed_log_return'] = np.log(data['close'] / data['close'].shift(1))

# Show the NaN values
data[data['processed_log_return'].isnull()]

,open,high,low,close,volume,original_close,processed_log_return
date,,,,,,,
2018-12-31,133.0,139.0,127.03,137.77,704259.41614,137.77,NaN


In [18]:
# Remove the NaN values
data = data[~data['processed_log_return'].isnull()]

In [19]:
# Initial Plot
fig = px.line(data, x=data.index, y="processed_log_return", title='Original BTCUSDT processed_close log_return')
fig.show()

## Outliers

Handling outliers could be done in many ways, but one common method for financial data is to employ Z-score.

In [20]:
# Compute Z-score for the 'close' price column
z_scores = np.abs((data['processed_log_return'] - data['processed_log_return'].mean()) / data['processed_log_return'].std())
print(f'Z-score mean: {z_scores.mean()}')
print(f'Z-score std: {z_scores.std()}')
print(f'Z-score max: {z_scores.max()}')
print(f'Z-score min: {z_scores.min()}')

Z-score mean: 0.6458351167310861
Z-score std: 0.7631036555790717
Z-score max: 12.050392636540483
Z-score min: 0.0004561194859027858


### Plotting the outliers

In [22]:
# Plot the outliers
mask = z_scores > 3
if mask.any():
	fig = px.scatter(x=data.index[mask], y=data.loc[mask, 'processed_log_return'], title='Outliers')
	fig.show()
else:
	print("No outliers with z-score > 3 found.")

### Adjusting the outliers

In [23]:
# Assuming df is your DataFrame and 'log_return' is the column with outliers
upper_bound = data['processed_log_return'].quantile(0.98)
print(f'Upper bound: {upper_bound}')
lower_bound = data['processed_log_return'].quantile(0.02)
print(f'Lower bound: {lower_bound}')

# Applying capping and flooring
data['outliers_processed_log_return'] = data['processed_log_return'].clip(lower=lower_bound, upper=upper_bound)


Upper bound: 0.09936647749342571
Lower bound: -0.09035647154696086


In [24]:
# Compute Z-score for the 'close' price column
z_scores = np.abs((data['outliers_processed_log_return'] - data['outliers_processed_log_return'].mean()) / data['outliers_processed_log_return'].std())
print(f'Z-score mean: {z_scores.mean()}')
print(f'Z-score std: {z_scores.std()}')
print(f'Z-score max: {z_scores.max()}')
print(f'Z-score min: {z_scores.min()}')

Z-score mean: 0.7530941920019362
Z-score std: 0.6573236202308199
Z-score max: 2.454855069578582
Z-score min: 0.0014117889235297028


In [25]:
# # Plot the outliers
# fig = px.scatter(data, x=data['outliers_processed_log_return'][z_scores > 3].index, y=data['outliers_processed_log_return'][z_scores > 3], title='Outliers')
# fig.show()

In [26]:
# Initial Plot
fig = px.line(data, x=data.index, y="outliers_processed_log_return", title='BTCUSDT Close outliers_processed_log_return')
fig.show()

In [27]:
data

,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return
date,,,,,,,,
2019-01-01,137.81,138.66,128.70,131.45,4.846048e+05,131.45,-0.046959,-0.046959
2019-01-02,131.45,140.67,130.00,139.10,4.017769e+05,139.10,0.056567,0.056567
2019-01-03,139.10,155.37,137.76,152.01,9.414863e+05,152.01,0.088753,0.088753
2019-01-04,152.01,153.64,143.11,146.30,7.080913e+05,146.30,-0.038287,-0.038287
2019-01-05,146.30,154.80,143.54,151.97,8.052489e+05,151.97,0.038024,0.038024
...,...,...,...,...,...,...,...,...
2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41,0.064027,0.064027
2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00,0.002174,0.002174
2020-12-31,732.00,758.74,716.62,752.17,1.008574e+06,752.17,0.027182,0.027182


## Stationarity Check (before normalization)

Many time series algorithms require the data to be stationary. The Augmented Dickey-Fuller test is often used for this purpose.

In [28]:
# Function to perform ADF test
def adf_test(series):
    result = adfuller(series, autolag='AIC')
    return result[1] <= 0.05  # p-value

# Check if the series is stationary
is_stationary = adf_test(data['outliers_processed_log_return'])

# If not stationary, difference the series
if not is_stationary:
    data['outliers_processed_log_return_diff'] = data['outliers_processed_log_return'].diff().dropna()

    # Plot after differencing
    fig = px.line(data, x=data.index, y="outliers_processed_log_return_diff", title='Differenced BTCUSDT outliers_processed_log_return')
    fig.show()

In [29]:
# Check that there is no missing data
data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

## Normalization

The data can be normalized using the MinMaxScaler or other methods like Z-score normalization, based on the downstream applications.

In [30]:
scaler = MinMaxScaler(feature_range=(-1, 1))
data[['normalized_outliers_processed_log_return']] = scaler.fit_transform(data[['outliers_processed_log_return']])

In [31]:
# Plot after normalization
fig = px.line(data, x=data.index, y="normalized_outliers_processed_log_return", title='Normalized BTCUSDT Close outliers_processed_log_return')
fig.show()

In [32]:
# Check that there is no missing data
data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

## Stationarity Check (after normalization)

Many time series algorithms require the data to be stationary. The Augmented Dickey-Fuller test is often used for this purpose.

In [33]:
# Function to perform ADF test
def adf_test(series):
    result = adfuller(series, autolag='AIC')
    return result[1] <= 0.05  # p-value

# Check if the series is stationary
is_stationary = adf_test(data['normalized_outliers_processed_log_return'])

# If not stationary, difference the series
if not is_stationary:
    data['normalized_outliers_processed_log_return_diff'] = data['normalized_outliers_processed_log_return'].diff().dropna()

    # Plot after differencing
    fig = px.line(data, x=data.index, y="normalized_outliers_processed_log_return_diff", title='Differenced BTCUSDT Close normalized_outliers_processed_log_return')
    fig.show()

In [34]:
# Check that there is no missing data
data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

## Denoising

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 733 entries, 2019-01-01 to 2021-01-02
Data columns (total 9 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   open                                      733 non-null    float64
 1   high                                      733 non-null    float64
 2   low                                       733 non-null    float64
 3   close                                     733 non-null    float64
 4   volume                                    733 non-null    float64
 5   original_close                            733 non-null    float64
 6   processed_log_return                      733 non-null    float64
 7   outliers_processed_log_return             733 non-null    float64
 8   normalized_outliers_processed_log_return  733 non-null    float64
dtypes: float64(9)
memory usage: 57.3 KB


### MODWT (Maximal Overlap Discrete Wavelet Transform)

Apply Discrete wavelet transform

In [42]:
wt = modwt(data['normalized_outliers_processed_log_return'], 'db2', 5)
wtmra = modwtmra(wt, 'db2')

In [43]:
df_wtmra = pd.DataFrame(wtmra)
df_wtmra = df_wtmra.transpose()

In [44]:
# Rename the df_wtrma columns
df_wtmra.columns = ['processed_log_return_wtmra_0', 'processed_log_return_wtmra_1', 'processed_log_return_wtmra_2', 'processed_log_return_wtmra_3', 'processed_log_return_wtmra_4', 'processed_log_return_wtmra_5']

In [45]:
df_wtmra['processed_log_return_wtmra_5_4'] = df_wtmra['processed_log_return_wtmra_5'] + df_wtmra['processed_log_return_wtmra_4']
df_wtmra['processed_log_return_wtmra_5_4_3'] = df_wtmra['processed_log_return_wtmra_5_4'] + df_wtmra['processed_log_return_wtmra_3']
df_wtmra['processed_log_return_wtmra_5_4_3_2'] = df_wtmra['processed_log_return_wtmra_5_4_3'] + df_wtmra['processed_log_return_wtmra_2']
df_wtmra['processed_log_return_wtmra_5_4_3_2_1'] = df_wtmra['processed_log_return_wtmra_5_4_3_2'] + df_wtmra['processed_log_return_wtmra_1']
df_wtmra['processed_log_return_wtmra_5_4_3_2_1_0'] = df_wtmra['processed_log_return_wtmra_5_4_3_2_1'] + df_wtmra['processed_log_return_wtmra_0']

In [46]:
df_wtmra['processed_log_return_wtmra_0_1'] = df_wtmra['processed_log_return_wtmra_0'] + df_wtmra['processed_log_return_wtmra_1']
df_wtmra['processed_log_return_wtmra_0_1_2'] = df_wtmra['processed_log_return_wtmra_0_1'] + df_wtmra['processed_log_return_wtmra_2']
df_wtmra['processed_log_return_wtmra_0_1_2_3'] = df_wtmra['processed_log_return_wtmra_0_1_2'] + df_wtmra['processed_log_return_wtmra_3']
df_wtmra['processed_log_return_wtmra_0_1_2_3_4'] = df_wtmra['processed_log_return_wtmra_0_1_2_3'] + df_wtmra['processed_log_return_wtmra_4']
df_wtmra['processed_log_return_wtmra_0_1_2_3_4_5'] = df_wtmra['processed_log_return_wtmra_0_1_2_3_4'] + df_wtmra['processed_log_return_wtmra_5']

In [47]:
# Add the data['date'] column to the df_wtmra DataFrame
df_wtmra['date'] = data.index
# Set the index of df_wtmra to be the date column
df_wtmra.set_index('date', inplace=True)

In [48]:
# Check that there is no missing data
df_wtmra.index = pd.to_datetime(df_wtmra.index)
my_range_wtmra = pd.date_range(start=df_wtmra.index.min(), end=df_wtmra.index.max(), freq=ref_data_range)
my_range_wtmra[~my_range_wtmra.isin(df_wtmra.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [49]:
df_wtmra.head()

,processed_log_return_wtmra_0,processed_log_return_wtmra_1,processed_log_return_wtmra_2,processed_log_return_wtmra_3,processed_log_return_wtmra_4,processed_log_return_wtmra_5,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,
2019-01-01,-0.388493,-0.154228,-0.122490,0.077314,0.064969,-0.019590,0.045378,0.122693,0.000202,-0.154025,-0.542518,-0.542721,-0.665211,-0.587897,-0.522928,-0.542518
2019-01-02,0.179795,0.248564,0.016403,0.070397,0.057980,-0.024321,0.033658,0.104055,0.120458,0.369022,0.548817,0.428359,0.444761,0.515158,0.573138,0.548817
2019-01-03,0.412090,0.269382,0.130709,0.055441,0.049682,-0.029186,0.020495,0.075937,0.206646,0.476028,0.888118,0.681472,0.812181,0.867623,0.917304,0.888118
2019-01-04,-0.590937,-0.068419,0.171862,0.031188,0.039436,-0.034232,0.005205,0.036393,0.208256,0.139837,-0.451100,-0.659356,-0.487493,-0.456305,-0.416868,-0.451100
2019-01-05,0.306014,-0.109709,0.159339,0.009100,0.027945,-0.039345,-0.011400,-0.002300,0.157040,0.047331,0.353345,0.196305,0.355645,0.364745,0.392690,0.353345


In [50]:
df_wtmra.tail()

,processed_log_return_wtmra_0,processed_log_return_wtmra_1,processed_log_return_wtmra_2,processed_log_return_wtmra_3,processed_log_return_wtmra_4,processed_log_return_wtmra_5,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,
2020-12-29,0.117949,0.152358,0.189280,0.071674,0.092662,0.003536,0.096198,0.167872,0.357152,0.509510,0.627458,0.270306,0.459586,0.531260,0.623922,0.627458
2020-12-30,-0.256800,0.011849,0.047390,0.084140,0.089614,-0.000760,0.088854,0.172994,0.220384,0.232233,-0.024568,-0.244951,-0.197561,-0.113422,-0.023807,-0.024568
2020-12-31,0.207854,-0.029465,-0.107457,0.088588,0.084935,-0.005404,0.079532,0.168120,0.060663,0.031198,0.239052,0.178389,0.070932,0.159521,0.244456,0.239052
2021-01-01,-0.122013,-0.087864,-0.219130,0.090081,0.078548,-0.010194,0.068354,0.158435,-0.060694,-0.148558,-0.270571,-0.209877,-0.429006,-0.338925,-0.260377,-0.270571
2021-01-02,0.177894,-0.265514,-0.209627,0.084728,0.071895,-0.014922,0.056973,0.141700,-0.067927,-0.333440,-0.155546,-0.087619,-0.297247,-0.212519,-0.140624,-0.155546


In [51]:
# Reconstruct the series using wtmra coefficients
# reconstructed_series = np.sum(wtmra, axis=0)

# Reconstruct multiples series by incrementing the number of coefficients used by addind on top of the last one

reconstructed_series = []

# loop through the coefficients reverser order
for i in range(len(wtmra)-1, -1, -1):
    reconstructed_series.append(np.sum(wtmra[:i], axis=0))

# for i in range(0, len(wtmra)):
    
#     reconstructed_series.append(np.sum(wtmra[:i], axis=0))

# Plot the reconstructed series
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['normalized_outliers_processed_log_return'], name='Original'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4'], name='wtmra_5_4'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3'], name='wtmra_5_4_3'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2'], name='wtmra_5_4_3_2'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2_1'], name='wtmra_5_4_3_2_1'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2_1_0'], name='wtmra_5_4_3_2_1_0'))

fig.update_layout(title='Reconstructed BTCUSDT Close normalized_outliers_processed_log_return', xaxis_title='Date', yaxis_title='Close normalized_outliers_processed_log_return')
fig.show()


In [52]:
# Reconstruct the series using wtmra coefficients
# reconstructed_series = np.sum(wtmra, axis=0)

# Reconstruct multiples series by incrementing the number of coefficients used by addind on top of the last one

reconstructed_series = []

# loop through the coefficients reverser order
for i in range(len(wtmra)-1, -1, -1):
    reconstructed_series.append(np.sum(wtmra[:i], axis=0))

# for i in range(0, len(wtmra)):
    
#     reconstructed_series.append(np.sum(wtmra[:i], axis=0))

# Plot the reconstructed series
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['normalized_outliers_processed_log_return'], name='Original'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0_1'], name='wtmra_0_1'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0_1_2'], name='wtmra_0_1_2'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0_1_2_3'], name='wtmra_0_1_2_3'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0_1_2_3_4'], name='wtmra_0_1_2_3_4'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0_1_2_3_4_5'], name='wtmra_0_1_2_3_4_5'))

fig.update_layout(title='Reconstructed BTCUSDT Close normalized_outliers_processed_log_return', xaxis_title='Date', yaxis_title='Close normalized_outliers_processed_log_return')
fig.show()


In [53]:
# Plot the wtmra coefficients
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5'], name='wtmra_5'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_4'], name='wtmra_4'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_3'], name='wtmra_3'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_2'], name='wtmra_2'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_1'], name='wtmra_1'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_0'], name='wtmra_0'))

# Add the original series
fig.add_trace(go.Scatter(x=data.index, y=data['normalized_outliers_processed_log_return'], name='Original'))

# Add the reconstructed series
fig.add_trace(go.Scatter(x=data.index, y=reconstructed_series, name='Reconstructed'))

fig.update_layout(title='MODWT Coefficients', xaxis_title='Time', yaxis_title='Magnitude')
fig.show()

In [54]:
# Add df_wtmra colums to the data DataFrame by joining on the date index
data = data.join(df_wtmra)

In [55]:
# Check that there is no missing data
data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

## Remove #N/A and save to CSV

Finally, we remove any remaining #N/A values and save the data to a CSV file.

In [56]:
# Remove NANs
data.normalized_outliers_processed_log_return.dropna(inplace=True)

In [57]:
# Check that there is no missing data
data.index = pd.to_datetime(data.index)
my_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [58]:
# Get data from_date until until_date inclusive
data = data[(data.index >= from_date) & (data.index <= until_date)]

In [59]:
data.head()

,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,processed_log_return_wtmra_0,...,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,,,,,,
2019-01-01,137.81,138.66,128.70,131.45,484604.83962,131.45,-0.046959,-0.046959,-0.542518,-0.388493,...,0.045378,0.122693,0.000202,-0.154025,-0.542518,-0.542721,-0.665211,-0.587897,-0.522928,-0.542518
2019-01-02,131.45,140.67,130.00,139.10,401776.88797,139.10,0.056567,0.056567,0.548817,0.179795,...,0.033658,0.104055,0.120458,0.369022,0.548817,0.428359,0.444761,0.515158,0.573138,0.548817
2019-01-03,139.10,155.37,137.76,152.01,941486.31524,152.01,0.088753,0.088753,0.888118,0.412090,...,0.020495,0.075937,0.206646,0.476028,0.888118,0.681472,0.812181,0.867623,0.917304,0.888118
2019-01-04,152.01,153.64,143.11,146.30,708091.31714,146.30,-0.038287,-0.038287,-0.451100,-0.590937,...,0.005205,0.036393,0.208256,0.139837,-0.451100,-0.659356,-0.487493,-0.456305,-0.416868,-0.451100
2019-01-05,146.30,154.80,143.54,151.97,805248.93821,151.97,0.038024,0.038024,0.353345,0.306014,...,-0.011400,-0.002300,0.157040,0.047331,0.353345,0.196305,0.355645,0.364745,0.392690,0.353345


In [60]:
data.tail()

,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,processed_log_return_wtmra_0,...,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,,,,,,
2020-12-27,626.78,652.91,615.26,637.44,9.585855e+05,637.44,0.016801,0.016801,0.129618,-0.221445,...,0.101670,0.117578,0.339745,0.351063,0.129618,-0.210127,0.012040,0.027948,0.119078,0.129618
2020-12-28,637.44,717.13,625.00,685.11,1.859968e+06,685.11,0.072119,0.072119,0.712768,0.164158,...,0.100931,0.145761,0.415573,0.548610,0.712768,0.297195,0.567008,0.611838,0.705467,0.712768
2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41,0.064027,0.064027,0.627458,0.117949,...,0.096198,0.167872,0.357152,0.509510,0.627458,0.270306,0.459586,0.531260,0.623922,0.627458
2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00,0.002174,0.002174,-0.024568,-0.256800,...,0.088854,0.172994,0.220384,0.232233,-0.024568,-0.244951,-0.197561,-0.113422,-0.023807,-0.024568
2020-12-31,732.00,758.74,716.62,752.17,1.008574e+06,752.17,0.027182,0.027182,0.239052,0.207854,...,0.079532,0.168120,0.060663,0.031198,0.239052,0.178389,0.070932,0.159521,0.244456,0.239052


In [61]:
# Check if there is duplicate data_filled['date']
data[data.index.duplicated()]

,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,processed_log_return_wtmra_0,...,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,,,,,,


In [62]:
# Save the processed data
data.to_csv(f'../data/01-output-{data_name}-from-{from_date}-until-{until_date}-log-return.csv', index=True)

In [63]:
data_loaded = pd.read_csv(f'../data/01-output-{data_name}-from-{from_date}-until-{until_date}-log-return.csv', parse_dates=['date'], index_col='date')

In [64]:
my_range = pd.date_range(start=data_loaded.index.min(), end=data_loaded.index.max(), freq=ref_data_range)
my_range[~my_range.isin(data_loaded.index)]

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [65]:
# Plot the reconstructed series
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['normalized_outliers_processed_log_return'], name='Original'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4'], name='wtmra_5_4'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3'], name='wtmra_5_4_3'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2'], name='wtmra_5_4_3_2'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2_1'], name='wtmra_5_4_3_2_1'))
fig.add_trace(go.Scatter(x=df_wtmra.index, y=df_wtmra['processed_log_return_wtmra_5_4_3_2_1_0'], name='wtmra_5_4_3_2_1_0'))

fig.update_layout(title='Reconstructed BTCUSDT Close normalized_outliers_processed_log_return', xaxis_title='Date', yaxis_title='Close normalized_outliers_processed_log_return')
fig.show()